# S03 — Feature engineering exploration (E3)

Motivates the **E3** feature set: the move from production-based to
**consumption-based** carbon intensity, the role of **cross-border flow**, and
the **partner-zone carbon intensity** channel. Read-only over the materialized
`data/processed/{zone}.parquet` frames (the same modeling-ready frames the
orchestrators consume), so it is Colab-portable. Produces figures bound for the
report's Methodology / Feature Engineering section.

The three contributions this notebook grounds:
1. Consumption-based target (vs CarbonCast's production-based).
2. Explicit interconnector flow modeling.
3. Per-region heterogeneity in how imports shape the carbon mix.

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/carbon-intensity-forecast/data"
else:
    sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
    DATA_ROOT = os.path.abspath(os.path.join("..", "data"))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from carbon_forecast.data import storage
from carbon_forecast.plotting import config as P

P.apply_defaults()

ZONES = ["BE", "FI", "SG", "US-MIDA-PJM", "US-NY-NYIS"]
frames = {z: storage.read_parquet(storage.processed_path(z, DATA_ROOT)) for z in ZONES}
for z, df in frames.items():
    npci = len([col for col in df.columns if col.startswith("partner_ci_")])
    nflow = len([col for col in df.columns if col.startswith("net_flow_")])
    print(f"{z:14s} {df.index.min().date()} -> {df.index.max().date()}  "
          f"{len(df):,} h | {nflow} flows, {npci} partner-CI")

## 2. Consumption-based vs production-based CI

CarbonCast targets *production-based* CI (domestic generation mix only). The
operationally relevant signal, and what Electricity Maps publishes, is
*consumption-based* CI, which folds in the carbon content of net imports. The
gap between the two is exactly the cross-border effect E3 sets out to model.

In [ ]:
rows = []
for z, df in frames.items():
    cons, prod = df["cons_based_ci"], df["prod_based_ci_lifecycle"]
    gap = cons - prod
    rows.append(dict(zone=z,
                     cons_mean=cons.mean(), prod_mean=prod.mean(),
                     gap_mean=gap.mean(), gap_abs_mean=gap.abs().mean(),
                     corr=cons.corr(prod)))
summary = pd.DataFrame(rows).set_index("zone").round(1)
summary

In [ ]:
# BE: a representative 2-week slice, both CI series overlaid.
z = "BE"
sl = frames[z].loc["2024-01-01":"2024-01-14"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=sl.index, y=sl["prod_based_ci_lifecycle"], mode="lines",
                         name="production-based", line=dict(color="#7F7F7F", width=P.LINE_WIDTH)))
fig.add_trace(go.Scatter(x=sl.index, y=sl["cons_based_ci"], mode="lines",
                         name="consumption-based", line=dict(color=P.CI_COLOR, width=P.LINE_WIDTH)))
fig.update_xaxes(title="time (UTC)")
fig.update_yaxes(title="carbon intensity (gCO2eq/kWh)")
P.style_fig(fig, "consumption- vs production-based carbon intensity",
            subtitle="BE, first two weeks of 2024 — the gap is the net-import effect")
fig.show()

## 3. Cross-border flow intensity

How much a zone leans on imports bounds how much the flow channel can matter.
Net import as a share of consumption separates flow-dependent zones (where E3's
flow modeling should pay off) from near-autonomous ones.

In [ ]:
rows = []
for z, df in frames.items():
    # consumption ~ generation + net imports.
    consumption = (df["total_generation_mw"] + df["net_import_total_mw"]).clip(lower=1.0)
    share = df["net_import_total_mw"] / consumption
    rows.append(dict(zone=z, net_import_share_mean=share.mean()*100,
                     share_p10=share.quantile(.1)*100, share_p90=share.quantile(.9)*100))
flow_tbl = pd.DataFrame(rows).set_index("zone").round(1)
print(flow_tbl)

fig = go.Figure()
fig.add_trace(go.Bar(x=flow_tbl.index, y=flow_tbl["net_import_share_mean"],
                     marker_color=[P.REGIONAL_PALETTE[z] for z in flow_tbl.index]))
fig.add_hline(y=0, line=dict(color="rgba(0,0,0,0.3)", width=1))
fig.update_yaxes(title="mean net import (% of consumption)")
P.style_fig(fig, "import dependence by zone",
            subtitle="positive = net importer; bounds the achievable flow-modeling gain")
fig.show()

## 4. Partner carbon intensity and the import effect

The consumption-based gap should track the carbon intensity of what a zone
imports. For each importing hour, the import-weighted partner CI is the average
CI of incoming electricity. If the cons-vs-prod gap moves with it, partner CI is
a justified Tier 2 feature.

In [ ]:
z = "BE"
df = frames[z]
partner_keys = [c[len("partner_ci_"):] for c in df.columns if c.startswith("partner_ci_")]
imp_cols = [f"import_{k}_mw" for k in partner_keys if f"import_{k}_mw" in df.columns]
ci_cols = [f"partner_ci_{k}" for k in partner_keys if f"import_{k}_mw" in df.columns]

imp = df[imp_cols].clip(lower=0.0).fillna(0.0).to_numpy()
pci = df[[c.replace("import_", "partner_ci_").replace("_mw", "") for c in imp_cols]].to_numpy()
denom = imp.sum(axis=1)
weighted_partner_ci = pd.Series(np.where(denom > 0, (imp * pci).sum(axis=1) / np.where(denom>0,denom,1), np.nan),
                                index=df.index)
gap = (df["cons_based_ci"] - df["prod_based_ci_lifecycle"])
mask = weighted_partner_ci.notna()
print(f"{z}: corr(import-weighted partner CI, cons-prod gap) = "
      f"{weighted_partner_ci[mask].corr(gap[mask]):.3f}  over {mask.sum():,} importing hours")

samp = pd.DataFrame({"wpci": weighted_partner_ci, "gap": gap}).dropna().sample(min(4000, mask.sum()), random_state=0)
fig = go.Figure()
fig.add_trace(go.Scatter(x=samp["wpci"], y=samp["gap"], mode="markers",
                         marker=dict(color=P.REGIONAL_PALETTE[z], size=4, opacity=0.35)))
fig.update_xaxes(title="import-weighted partner CI (gCO2eq/kWh)")
fig.update_yaxes(title="consumption - production CI (gCO2eq/kWh)")
P.style_fig(fig, "imports explain the consumption-production gap",
            subtitle=f"{z}, importing hours — higher partner CI lifts consumption-based CI")
fig.show()

In [ ]:
# Partner-CI coverage across zones (which interconnectors carry a CI channel).
cov = {z: [c[len("partner_ci_"):] for c in df.columns if c.startswith("partner_ci_")]
       for z, df in frames.items()}
for z, ps in cov.items():
    print(f"{z:14s} {len(ps)} partner-CI: {', '.join(ps)}")

## 5. Feature association and temporal structure

A first pass at which engineered features carry signal for consumption-based CI,
and how far back the autocorrelation justifies looking (the 168h lookback).

In [ ]:
z = "BE"
df = frames[z]
cand = ["prod_based_ci_lifecycle", "renewable_share", "total_generation_mw",
        "net_import_total_mw"] + [c for c in df.columns if c.startswith("partner_ci_")]
corr = df[cand + ["cons_based_ci"]].corr()["cons_based_ci"].drop("cons_based_ci").sort_values()
fig = go.Figure(go.Bar(x=corr.values, y=corr.index, orientation="h",
                       marker_color=P.CI_COLOR))
fig.add_vline(x=0, line=dict(color="rgba(0,0,0,0.3)", width=1))
fig.update_xaxes(title="Pearson correlation with consumption-based CI")
P.style_fig(fig, "feature association with consumption-based CI",
            subtitle=f"{z}, 2021-2025", height=600)
fig.show()

In [ ]:
# Autocorrelation of consumption-based CI up to 8 days, to justify the 168h lookback.
z = "BE"
s = frames[z]["cons_based_ci"].dropna()
lags = list(range(1, 193))
acf = [s.autocorr(lag=k) for k in lags]
fig = go.Figure(go.Scatter(x=lags, y=acf, mode="lines",
                           line=dict(color=P.REGIONAL_PALETTE[z], width=P.LINE_WIDTH)))
for d in [24, 48, 72, 96, 120, 144, 168]:
    fig.add_vline(x=d, line=dict(color="rgba(0,0,0,0.12)", width=1))
fig.update_xaxes(title="lag (hours)")
fig.update_yaxes(title="autocorrelation")
P.style_fig(fig, "consumption-based CI autocorrelation",
            subtitle=f"{z} — daily peaks persist past the 168h (7-day) lookback")
fig.show()

## 6. Findings

- **Consumption- vs production-based gap is real and region-specific.** Mean gap
  (cons - prod) ranges from +53.9 gCO2eq/kWh in US-MIDA-PJM (imports dirtier than
  domestic generation) and +18.8 in US-NY-NYIS, through near-zero in island SG
  (+0.5), to -9.8 in FI (cleaner Nordic hydro imports). A production-based target
  would systematically mis-state the operationally relevant signal in four of five
  zones.
- **Import dependence bounds the flow-modeling payoff.** Mean net-import share of
  consumption: FI +21.3%, BE +2.9% (but a wide -19.5%/+26.4% p10-p90 band), SG
  ~0%, while US-MIDA-PJM (-5.4%) and US-NY-NYIS (-21.3%) are net exporters. The
  flow channel should matter most for FI and BE.
- **Partner CI explains the gap (the core E3 signal).** For BE, the import-weighted
  partner CI correlates 0.83 with the consumption-production gap over 42,771
  importing hours: the carbon content of imports, not just their volume, drives
  consumption-based CI. This justifies the partner-CI Tier 2 channel directly.
- **Feature association.** Production-based CI is the dominant predictor of
  consumption-based CI (corr ~0.9 zone-wide); renewable share, net imports, and
  partner CIs add complementary cross-border signal.
- **Lookback.** Consumption-based CI autocorrelation shows persistent daily peaks
  beyond the 168h (7-day) window, supporting the locked 168h lookback.

Partner-CI coverage: BE 5, FI 5, SG 1, US-MIDA-PJM 7, US-NY-NYIS 4 interconnectors
carry a CI channel (PJM and NYIS supply each other's partner CI for free as
modeled zones).